# LEGACY — DO NOT EXECUTE FOR CURRENT EXPERIMENT

Historical notebook `03b_inspect_features_windows.ipynb` for retired protocol `features-0.2.0`. Its references to ventanas globales, primary_stride=1 y strides 1, 10, 30, 65 are preserved only for audit traceability and are not valid inputs or recommendations for `common-core-1.0.0`.

This notebook contains no executable cells and is not part of the canonical pipeline. Current sources of truth are `configs/experiment.yaml`, `COMMON_CORE_CERTIFICATION.md`, `scripts/` and `tests/`.


> **LEGACY HISTORICAL MATERIAL — NOT CURRENT PROTOCOL**

## Controles

| Control | Default | Uso |
|---|---|---|
| `TICKER` | `"NVDA"` | Serie `log_return`, densidades, ventana ejemplo |
| `STRIDE` | `1` | Parquet de ventanas a inspeccionar (`primary_stride` / default recomendado) |
| `STRIDE_B` | `65` | Segunda ventana (comparar solape / bloques disjuntos) |
| `SAVE_FIGS` | `False` | Si `True`, exporta PNGs a `notebooks/figures/03b_*` |

`STRIDE=1` es el **default recomendado** comparable del equipo; no es el único dataset del menú.


> **LEGACY CODE — NON-EXECUTABLE; NOT CURRENT PROTOCOL**

The historical Python source is retained as inert Markdown for traceability only.

````python
from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- Controles ---
TICKER = "NVDA"
STRIDE = 1          # primary / default recomendado
STRIDE_B = 65       # segunda ventana (bloques casi disjuntos)
SAVE_FIGS = False

EXPECTED_VERSION = "features-0.2.0"
EXPECTED_STRIDES = [1, 10, 30, 65]
PRIMARY_STRIDE = 1

cwd = Path.cwd().resolve()
ROOT = cwd if (cwd / "data" / "features").is_dir() else cwd.parent
FEATURES_DIR = ROOT / "data" / "features"
FIG_DIR = ROOT / "notebooks" / "figures"

assert FEATURES_DIR.is_dir(), f"No se encuentra data/features en {ROOT}"
print(f"ROOT = {ROOT}")
print(f"TICKER={TICKER} | STRIDE={STRIDE} | STRIDE_B={STRIDE_B} | SAVE_FIGS={SAVE_FIGS}")
````


> **LEGACY HISTORICAL MATERIAL — NOT CURRENT PROTOCOL**

## 1. Contexto del paso 3 (multi-stride)

```
data/clean/ohlcv_clean.parquet
        │
        │  3 features diarias por ticker (mismas fórmulas que 0.1.0)
        ▼
data/features/daily_features.parquet
        │
        │  ventanas T=65, filas consecutivas, un ticker por ventana
        │  MENÚ de strides (ficheros separados — no mezclar en un train sin acuerdo)
        ▼
data/features/windows_65_stride1.parquet   ← primary_stride=1 (default recomendado)
data/features/windows_65_stride10.parquet
data/features/windows_65_stride30.parquet
data/features/windows_65_stride65.parquet
```

**Qué es el stride?** Cuántas filas de features avanzas al crear la siguiente ventana.
Con `T=65` y `stride=1` las ventanas se solapan en 64 pasos; con `stride=65` son bloques
sin solape (disjuntos en el eje de filas).


> **LEGACY HISTORICAL MATERIAL — NOT CURRENT PROTOCOL**

## 2. Verificación SHA + manifest

Leemos `checksums.sha256` y comprobamos que los artefactos en disco coinciden.
Si `data_version != features-0.2.0` o faltan strides → **PARA**.


> **LEGACY CODE — NON-EXECUTABLE; NOT CURRENT PROTOCOL**

The historical Python source is retained as inert Markdown for traceability only.

````python
def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

checksums_path = FEATURES_DIR / "checksums.sha256"
manifest_path = FEATURES_DIR / "features_manifest.json"
assert checksums_path.is_file() and manifest_path.is_file()

expected = {}
for line in checksums_path.read_text(encoding="utf-8").splitlines():
    line = line.strip()
    if not line:
        continue
    digest, rel = line.split(None, 1)
    expected[rel.strip()] = digest

required_rels = [
    "data/features/daily_features.parquet",
    *[f"data/features/windows_65_stride{s}.parquet" for s in EXPECTED_STRIDES],
]
missing = [r for r in required_rels if r not in expected]
if missing:
    raise RuntimeError(f"Faltan entradas en checksums.sha256: {missing}")

sha_rows = []
for rel in required_rels:
    path = ROOT / rel
    actual = sha256_file(path)
    ok = actual == expected[rel]
    sha_rows.append({"artifact": Path(rel).name, "path": rel, "expected": expected[rel], "actual": actual, "ok": ok})

sha_df = pd.DataFrame(sha_rows)
display(sha_df)

if not sha_df["ok"].all():
    raise RuntimeError("SHA256 NO coincide con checksums.sha256. PARAR.")

print("SHA daily + windows_65_stride{1,10,30,65}: OK")
````


> **LEGACY CODE — NON-EXECUTABLE; NOT CURRENT PROTOCOL**

The historical Python source is retained as inert Markdown for traceability only.

````python
with manifest_path.open(encoding="utf-8") as f:
    manifest = json.load(f)

version = manifest.get("data_version")
if version != EXPECTED_VERSION:
    raise RuntimeError(f"data_version={version!r} != {EXPECTED_VERSION!r}. PARAR.")

wp = manifest["window_params"]
strides = list(wp["strides"])
primary = int(wp["primary_stride"])
if strides != EXPECTED_STRIDES:
    raise RuntimeError(f"strides={strides} != {EXPECTED_STRIDES}. PARAR.")
if primary != PRIMARY_STRIDE:
    raise RuntimeError(f"primary_stride={primary} != {PRIMARY_STRIDE}. PARAR.")

# Ficheros por stride deben existir
for s in strides:
    p = FEATURES_DIR / f"windows_65_stride{s}.parquet"
    if not p.is_file():
        raise RuntimeError(f"Falta parquet de stride: {p}. PARAR.")

print(f"data_version   : {version}")
print(f"built_at_utc   : {manifest.get('built_at_utc')}")
print(f"T              : {wp['T']}")
print(f"strides        : {strides}")
print(f"primary_stride : {primary}  ← DEFAULT RECOMENDADO comparable")
print(f"channel_order  : {wp['channel_order']}")
print(f"tensor_shape   : {wp['tensor_shape']}")
print(f"n_windows_formula: {wp.get('n_windows_formula')}")
print()
print("--- Fórmulas ---")
for k, v in manifest["formulas"].items():
    print(f"  {k}: {v}")
print()
print("--- Políticas ---")
for key in ("volume_eq_0", "high_eq_low", "gaps_vs_union_calendar", "winsorize",
            "standardize_or_nvda_calibration", "mixed_strides_in_one_train"):
    if key in manifest["policies"]:
        print(f"  {key}: {manifest['policies'][key]}")
print()
print(f"n_features_rows = {manifest['n_features_rows']}")
print(f"n_windows_total_by_stride = {manifest['n_windows_total_by_stride']}")
````


> **LEGACY HISTORICAL MATERIAL — NOT CURRENT PROTOCOL**

## 3. Panel diario de features

| Columna | Significado |
|---|---|
| `date` | Día de la fila de features |
| `ticker` | Símbolo |
| `log_return` | $\ln(Close_t / Close_{t-1})$ |
| `log_high_low_range` | $\ln(High_t / Low_t)$; si `High==Low` → `0.0` |
| `log_volume` | $\ln(Volume_t)$; filas `Volume==0` ya dropeadas |

Sin ffill / winsorize / estandarizar. Las ventanas usan filas consecutivas de este panel.


> **LEGACY CODE — NON-EXECUTABLE; NOT CURRENT PROTOCOL**

The historical Python source is retained as inert Markdown for traceability only.

````python
daily = pd.read_parquet(FEATURES_DIR / "daily_features.parquet")
print("shape:", daily.shape)
print("dtypes:")
print(daily.dtypes)
print()
print("--- head ---")
display(daily.head())
print("--- tail ---")
display(daily.tail())
````


> **LEGACY CODE — NON-EXECUTABLE; NOT CURRENT PROTOCOL**

The historical Python source is retained as inert Markdown for traceability only.

````python
counts_daily = daily["ticker"].value_counts().sort_index().rename("n_rows")
manifest_daily = pd.Series(manifest["n_features_rows_by_ticker"]).sort_index().rename("manifest")
cmp_daily = pd.concat([counts_daily, manifest_daily], axis=1)
cmp_daily["match"] = cmp_daily["n_rows"] == cmp_daily["manifest"]
display(cmp_daily)
print(f"total = {len(daily)} | manifest = {manifest['n_features_rows']}")
assert cmp_daily["match"].all() and len(daily) == manifest["n_features_rows"]
print("Conteos panel diario alinean con manifest: OK")
````


> **LEGACY HISTORICAL MATERIAL — NOT CURRENT PROTOCOL**

### 3.1 Serie de `log_return` (`TICKER`)


> **LEGACY CODE — NON-EXECUTABLE; NOT CURRENT PROTOCOL**

The historical Python source is retained as inert Markdown for traceability only.

````python
sub = daily.loc[daily["ticker"] == TICKER].sort_values("date")
assert len(sub) > 0, f"TICKER={TICKER!r} ausente"

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(sub["date"], sub["log_return"], lw=0.6, color="C0")
ax.axhline(0.0, color="gray", lw=0.8, alpha=0.7)
ax.set_title(f"log_return diario — {TICKER}")
ax.set_xlabel("date")
ax.set_ylabel("log_return")
ax.grid(True, alpha=0.3)
fig.tight_layout()
if SAVE_FIGS:
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    out = FIG_DIR / f"03b_log_return_{TICKER}.png"
    fig.savefig(out, dpi=120)
    print(f"guardado: {out.relative_to(ROOT)}")
plt.show()
````


> **LEGACY HISTORICAL MATERIAL — NOT CURRENT PROTOCOL**

### 3.2 Densidades de las 3 features (`TICKER`)


> **LEGACY CODE — NON-EXECUTABLE; NOT CURRENT PROTOCOL**

The historical Python source is retained as inert Markdown for traceability only.

````python
feat_cols = ["log_return", "log_high_low_range", "log_volume"]
sub = daily.loc[daily["ticker"] == TICKER, feat_cols]

fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))
for ax, col in zip(axes, feat_cols):
    ax.hist(sub[col].to_numpy(), bins=60, density=True, color="C0", alpha=0.75, edgecolor="none")
    ax.set_title(col)
    ax.set_xlabel(col)
    ax.set_ylabel("density")
    ax.grid(True, alpha=0.3)
fig.suptitle(f"Densidades empíricas — {TICKER} (n={len(sub)})", y=1.02)
fig.tight_layout()
if SAVE_FIGS:
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    out = FIG_DIR / f"03b_feature_densities_{TICKER}.png"
    fig.savefig(out, dpi=120, bbox_inches="tight")
    print(f"guardado: {out.relative_to(ROOT)}")
plt.show()
````


> **LEGACY HISTORICAL MATERIAL — NOT CURRENT PROTOCOL**

## 4. Comparativa de strides (menú de datasets)

Fórmula de conteo (por ticker):

```
n_windows(ticker, stride) = floor((n_rows - T) / stride) + 1   si n_rows >= T
                          = 0                                  si n_rows < T
```

Al **aumentar** el stride → **menos** ventanas y **menos solape**.
`stride=1` maximiza muestra (y correlación entre ventanas vecinas);
`stride=65` da bloques disjuntos (≈ n_rows/65 por ticker).


> **LEGACY CODE — NON-EXECUTABLE; NOT CURRENT PROTOCOL**

The historical Python source is retained as inert Markdown for traceability only.

````python
T = int(wp["T"])
channel_order = list(wp["channel_order"])
assert channel_order == ["log_return", "log_high_low_range", "log_volume"]

def expected_n_windows(n_rows: int, stride: int, t: int = T) -> int:
    if n_rows < t:
        return 0
    return (n_rows - t) // stride + 1

windows_by_stride: dict[int, pd.DataFrame] = {}
rows = []
for s in strides:
    path = FEATURES_DIR / f"windows_65_stride{s}.parquet"
    w = pd.read_parquet(path)
    windows_by_stride[s] = w
    n = len(w)
    n_man = manifest["n_windows_total_by_stride"][f"stride{s}"]
    # solape aproximado en filas: T - stride (si stride < T); 0 si stride >= T
    overlap = max(T - s, 0)
    rows.append({
        "stride": s,
        "path": str(path.relative_to(ROOT)),
        "n_windows": n,
        "manifest": n_man,
        "match": n == n_man,
        "overlap_rows": overlap,
        "primary": s == primary,
    })

cmp_stride = pd.DataFrame(rows).set_index("stride")
display(cmp_stride)
assert cmp_stride["match"].all()
print("Totales por stride alinean con manifest: OK")
````


> **LEGACY CODE — NON-EXECUTABLE; NOT CURRENT PROTOCOL**

The historical Python source is retained as inert Markdown for traceability only.

````python
# Tabla opcional: n_windows por ticker × stride
by_ticker = pd.DataFrame(
    {s: windows_by_stride[s]["ticker"].value_counts().sort_index() for s in strides}
)
by_ticker.columns = [f"stride{s}" for s in strides]
# cruzar con fórmula
for s in strides:
    expected = {
        t: expected_n_windows(int(n), s)
        for t, n in manifest["n_features_rows_by_ticker"].items()
    }
    got = by_ticker[f"stride{s}"].to_dict()
    assert got == expected, f"mismatch por ticker stride={s}"

display(by_ticker)
print("Conteos por ticker × stride alinean con la fórmula: OK")
print()
print("Interpretación rápida:")
print(f"  stride=1  → {cmp_stride.loc[1, 'n_windows']} ventanas (máximo solape: {T-1} filas)")
print(f"  stride=10 → {cmp_stride.loc[10, 'n_windows']} ventanas")
print(f"  stride=30 → {cmp_stride.loc[30, 'n_windows']} ventanas")
print(f"  stride=65 → {cmp_stride.loc[65, 'n_windows']} ventanas (sin solape en filas)")
````


> **LEGACY HISTORICAL MATERIAL — NOT CURRENT PROTOCOL**

## 5. Visualizar una ventana `[65, 3]`

Cada fila de `windows_65_stride*.parquet`:

| Campo | Rol |
|---|---|
| `ticker` | Símbolo |
| `window_start_date` / `window_end_date` | Extremos temporales de la ventana |
| `features_flat` | 195 floats → `reshape(65, 3)` row-major |

Canales: `log_return`, `log_high_low_range`, `log_volume`.


> **LEGACY CODE — NON-EXECUTABLE; NOT CURRENT PROTOCOL**

The historical Python source is retained as inert Markdown for traceability only.

````python
def windows_path(stride: int) -> Path:
    return FEATURES_DIR / f"windows_65_stride{stride}.parquet"

def pick_window(windows: pd.DataFrame, ticker: str, which: str = "first") -> pd.Series:
    block = windows.loc[windows["ticker"] == ticker].sort_values("window_end_date")
    assert len(block) > 0, f"sin ventanas para {ticker}"
    if which == "first":
        return block.iloc[0]
    if which == "middle":
        return block.iloc[len(block) // 2]
    if which == "last":
        return block.iloc[-1]
    raise ValueError(which)

def plot_window(row: pd.Series, title_prefix: str = ""):
    X = np.asarray(row["features_flat"], dtype=np.float64).reshape(T, 3)
    t = np.arange(T)
    fig, axes = plt.subplots(3, 1, figsize=(10, 6), sharex=True)
    for i, (ax, name) in enumerate(zip(axes, channel_order)):
        ax.plot(t, X[:, i], lw=1.0, color=f"C{i}")
        ax.set_ylabel(name)
        ax.grid(True, alpha=0.3)
    axes[-1].set_xlabel("paso t (0..64) dentro de la ventana")
    axes[0].set_title(
        f"{title_prefix}{row['ticker']} | "
        f"{row['window_start_date'].date()} → {row['window_end_date'].date()} | shape {X.shape}"
    )
    fig.tight_layout()
    return fig, X

assert STRIDE in strides, f"STRIDE={STRIDE} no está en {strides}"
w_main = windows_by_stride[STRIDE]
row_a = pick_window(w_main, TICKER, "first")
print(f"Usando {windows_path(STRIDE).relative_to(ROOT)} | n_windows={len(w_main)}")
fig_a, X_a = plot_window(row_a, title_prefix=f"Ventana A (stride={STRIDE}) — ")
if SAVE_FIGS:
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    out = FIG_DIR / f"03b_window_{TICKER}_stride{STRIDE}.png"
    fig_a.savefig(out, dpi=120)
    print(f"guardado: {out.relative_to(ROOT)}")
plt.show()
print(f"X_a shape={X_a.shape}; log_return mean={X_a[:,0].mean():.4f}; log_volume mean={X_a[:,2].mean():.4f}")
````


> **LEGACY HISTORICAL MATERIAL — NOT CURRENT PROTOCOL**

### 5.1 (Opcional) Segunda ventana con otro stride

Por defecto `STRIDE_B=65` (bloques disjuntos). Misma forma `[65,3]`; cambia el muestreo temporal.


> **LEGACY CODE — NON-EXECUTABLE; NOT CURRENT PROTOCOL**

The historical Python source is retained as inert Markdown for traceability only.

````python
assert STRIDE_B in strides, f"STRIDE_B={STRIDE_B} no está en {strides}"
w_b = windows_by_stride[STRIDE_B]
row_b = pick_window(w_b, TICKER, "first")
print(f"Usando {windows_path(STRIDE_B).relative_to(ROOT)} | n_windows={len(w_b)}")
fig_b, X_b = plot_window(row_b, title_prefix=f"Ventana B (stride={STRIDE_B}) — ")
plt.show()
print(f"X_b shape={X_b.shape}; mismas columnas/canales; distinto muestreo (stride={STRIDE_B})")
````


> **LEGACY HISTORICAL MATERIAL — NOT CURRENT PROTOCOL**

## 6. Check de consistencia (didáctico)

Para un `window_end_date` + ticker: `features_flat` → `reshape(65,3)` debe coincidir
con el slice de `daily_features` en esas 65 filas consecutivas.

Comprobamos **stride=1** y **otro stride** (`STRIDE_B`, default 65). No regeneramos features.


> **LEGACY CODE — NON-EXECUTABLE; NOT CURRENT PROTOCOL**

The historical Python source is retained as inert Markdown for traceability only.

````python
def consistency_check(windows: pd.DataFrame, stride: int, ticker: str, which: str = "middle") -> dict:
    row = pick_window(windows, ticker, which)
    X = np.asarray(row["features_flat"], dtype=np.float64).reshape(T, 3)
    panel_t = daily.loc[daily["ticker"] == row["ticker"]].sort_values("date").reset_index(drop=True)
    mask = (panel_t["date"] >= row["window_start_date"]) & (panel_t["date"] <= row["window_end_date"])
    slice_df = panel_t.loc[mask]
    Y = slice_df[channel_order].to_numpy(dtype=np.float64)

    n_ok = len(slice_df) == T
    shape_ok = X.shape == Y.shape == (T, 3)
    max_abs = float(np.max(np.abs(X - Y))) if shape_ok and n_ok else float("nan")
    tol = 1e-12
    status = "PASS" if (n_ok and shape_ok and max_abs <= tol) else "FAIL"
    return {
        "stride": stride,
        "ticker": row["ticker"],
        "window_start": row["window_start_date"].date().isoformat(),
        "window_end": row["window_end_date"].date().isoformat(),
        "n_slice": len(slice_df),
        "max_abs_diff": max_abs,
        "status": status,
        "X": X,
        "Y": Y,
        "slice_dates": slice_df["date"].to_numpy(),
    }

checks = [
    consistency_check(windows_by_stride[1], 1, TICKER, "middle"),
    consistency_check(windows_by_stride[STRIDE_B], STRIDE_B, TICKER, "middle"),
]
chk_df = pd.DataFrame(
    [{k: v for k, v in c.items() if k not in ("X", "Y", "slice_dates")} for c in checks]
)
display(chk_df)

for c in checks:
    print(
        f"stride={c['stride']}: max abs diff = {c['max_abs_diff']:.3e} → {c['status']} "
        f"({c['window_start']} → {c['window_end']})"
    )

# Vista rápida del check stride=1 (3 primeras filas)
c0 = checks[0]
cmp_head = pd.DataFrame(
    {
        "date": c0["slice_dates"][:3],
        **{f"win_{name}": c0["X"][:3, i] for i, name in enumerate(channel_order)},
        **{f"day_{name}": c0["Y"][:3, i] for i, name in enumerate(channel_order)},
    }
)
display(cmp_head)

if any(c["status"] != "PASS" for c in checks):
    raise AssertionError("Check de consistencia FAIL")
print(f"Check reshape vs daily_features: PASS (strides {{1, {STRIDE_B}}})")
````


> **LEGACY HISTORICAL MATERIAL — NOT CURRENT PROTOCOL**

## 7. Cierre

### Cómo lo usarán los compañeros

1. Elegir **un** parquet del menú, p.ej. `windows_65_stride1.parquet` (default recomendado)
   u otro stride si el experimento lo pide.
2. Cargar filas → `np.asarray(features_flat).reshape(65, 3)`.
3. **No mezclar** strides distintos en el mismo train sin acuerdo del equipo.

### Qué NO hace este notebook

- No regenera ni escribe `data/features/`.
- No hace splits train/val/test.
- No estandariza / calibra a NVDA.
- No entrena VAE / GAN / Diffusion / Ridge.
- No es un EDA estadístico profundo (eso fue 02c).


> **LEGACY CODE — NON-EXECUTABLE; NOT CURRENT PROTOCOL**

The historical Python source is retained as inert Markdown for traceability only.

````python
print("=" * 60)
print("RESUMEN RÁPIDO 03b (features-0.2.0 multi-stride)")
print("=" * 60)
print(f"data_version          : {manifest['data_version']}")
print(f"SHA daily+4 strides   : OK")
print(f"primary_stride        : {primary}")
print(f"n_features_rows       : {len(daily)}")
for s in strides:
    print(f"n_windows stride={s:<2}  : {len(windows_by_stride[s])}")
print(f"TICKER / STRIDE       : {TICKER} / {STRIDE}")
print(f"consistencia          : PASS (stride=1 y stride={STRIDE_B})")
````
